In [2]:
from pathlib import Path
import pandas as pd

# Data Columns
The data columns explanation is provided in the `notes.txt` file. We need to rename these columns using the mapping provided. First of all, we will check if all the information (columns) provided in the file are part of all the CSV data that we have

In [11]:
raw_data_folder = Path.cwd().parent/"data"/"raw"/"ligue1"

l1_data = {}

for file in raw_data_folder.glob("*.csv"):
    l1_data[file.stem] = pd.read_csv(file)

In [18]:
for name, df in l1_data.items():
    print(name, "has", len(df.columns), "columns")

F1_2122 has 105 columns
F1_2223 has 105 columns
F1_2324 has 105 columns
F1_2425 has 119 columns
F1_2526 has 131 columns


## Column mismatch
We can clearly see that the columns in the datasets are not the same, therefore we need to decide how to go forward

In [21]:
set(l1_data['F1_2122'].columns) == set(l1_data['F1_2223'].columns)

True

In [20]:
set(l1_data['F1_2122'].columns) == set(l1_data['F1_2526'].columns)

False

In [31]:
len(set(l1_data['F1_2122'].columns) - set(l1_data['F1_2526'].columns))

18

In [47]:
cols_sets = [set(df.columns) for df in l1_data.values()]
common_cols = set.intersection(*cols_sets[:4])
len(common_cols)

93

We notice that the first 3 datasets have the same set of columns, however we start to see a difference after 23-24 and it seems that there are removed not just added columns since the number of common columns is not equal to the number of columns of the smaller dataset

In [50]:
set(l1_data['F1_2122'].columns) - set(l1_data['F1_2425'].columns)


{'IWA',
 'IWCA',
 'IWCD',
 'IWCH',
 'IWD',
 'IWH',
 'VCA',
 'VCCA',
 'VCCD',
 'VCCH',
 'VCD',
 'VCH'}

In [51]:
set(l1_data['F1_2425'].columns) - set(l1_data['F1_2122'].columns)

{'1XBA',
 '1XBCA',
 '1XBCD',
 '1XBCH',
 '1XBD',
 '1XBH',
 'BFA',
 'BFCA',
 'BFCD',
 'BFCH',
 'BFD',
 'BFE<2.5',
 'BFE>2.5',
 'BFEA',
 'BFEAHA',
 'BFEAHH',
 'BFEC<2.5',
 'BFEC>2.5',
 'BFECA',
 'BFECAHA',
 'BFECAHH',
 'BFECD',
 'BFECH',
 'BFED',
 'BFEH',
 'BFH'}

## Interpretation
What we can do is set a specific set of bookmakers that is present for all datasets that we are going to use for the purpose of this project. Later we can check if those specific bookmakers are enough to represent the actual average or how good it approximates it.

An important thing to note here is that we also have **handicap** odds in our data. We are going to ignore the handicap odds and the goals difference odds for the first phase of the project.

In [52]:
common_cols = set.intersection(*cols_sets)

In [58]:
l1_data['F1_2324'][list(common_cols)]

,PCAHA,B365CAHH,AvgCD,MaxCAHH,Avg<2.5,HTAG,B365CAHA,HTR,AvgA,AvgAHA,...,HY,P>2.5,AvgCAHA,PC<2.5,B365D,Div,HST,PCAHH,BWCH,AwayTeam
0,2.14,1.81,3.40,1.86,1.94,0,2.13,H,2.76,2.01,...,3,1.93,2.07,1.99,3.50,F1,3,1.79,2.45,Lille
1,1.92,2.03,3.69,2.03,2.05,1,1.90,D,4.82,1.96,...,1,1.81,1.96,2.05,4.00,F1,4,2.01,1.95,Reims
2,1.93,2.00,4.77,2.00,2.60,0,1.93,D,8.53,1.91,...,0,1.53,1.93,2.08,5.50,F1,4,1.99,1.40,Lorient
3,1.88,2.02,3.26,2.11,1.82,2,1.88,A,1.96,1.96,...,2,2.05,1.84,1.66,3.50,F1,8,2.05,3.50,Lens
4,2.09,1.82,3.48,1.89,2.13,2,2.08,A,2.04,2.04,...,0,1.75,2.04,2.08,3.60,F1,7,1.85,2.90,Monaco
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
301,1.99,1.86,5.73,1.92,2.90,0,2.07,H,6.94,1.99,...,1,1.41,2.02,2.86,4.75,F1,6,1.92,1.31,Strasbourg
302,2.00,1.88,3.91,1.94,2.82,2,2.05,A,1.60,1.95,...,2,1.44,1.96,2.54,4.00,F1,2,1.90,4.33,Paris SG
303,1.98,1.93,4.89,1.93,2.92,0,2.00,H,6.87,1.82,...,1,1.40,2.05,3.04,5.00,F1,6,1.93,1.53,Nantes
304,1.96,1.94,3.63,1.95,2.36,0,1.99,D,2.46,1.81,...,1,1.60,2.00,2.60,3.30,F1,4,1.94,2.95,Rennes


Let's check the common bookmakers we have in these columns

In [133]:
non_bookies = [
    'Div',
    'Date',
    'Time',
    'HomeTeam',
    'AwayTeam',
    'FTHG',
    'FTAG',
    'FTR',
    'HTHG',
    'HTAG',
    'HTR',
    'HS',
    'AS',
    'HST',
    'AST',
    'HC',
    'AC',
    'HF',
    'AF',
    'HY',
    'AY',
    'HR',
    'AR',
]

In [134]:
set(non_bookies).issubset(common_cols)

True

In [138]:
common_cols - set(non_bookies)

{'AHCh',
 'AHh',
 'Avg<2.5',
 'Avg>2.5',
 'AvgA',
 'AvgAHA',
 'AvgAHH',
 'AvgC<2.5',
 'AvgC>2.5',
 'AvgCA',
 'AvgCAHA',
 'AvgCAHH',
 'AvgCD',
 'AvgCH',
 'AvgD',
 'AvgH',
 'B365<2.5',
 'B365>2.5',
 'B365A',
 'B365AHA',
 'B365AHH',
 'B365C<2.5',
 'B365C>2.5',
 'B365CA',
 'B365CAHA',
 'B365CAHH',
 'B365CD',
 'B365CH',
 'B365D',
 'B365H',
 'BWA',
 'BWCA',
 'BWCD',
 'BWCH',
 'BWD',
 'BWH',
 'Max<2.5',
 'Max>2.5',
 'MaxA',
 'MaxAHA',
 'MaxAHH',
 'MaxC<2.5',
 'MaxC>2.5',
 'MaxCA',
 'MaxCAHA',
 'MaxCAHH',
 'MaxCD',
 'MaxCH',
 'MaxD',
 'MaxH',
 'P<2.5',
 'P>2.5',
 'PAHA',
 'PAHH',
 'PC<2.5',
 'PC>2.5',
 'PCAHA',
 'PCAHH',
 'PSA',
 'PSCA',
 'PSCD',
 'PSCH',
 'PSD',
 'PSH'}

In [158]:
non_bookies_cols = {
    'Div' : "league_division",
    'Date' : "match_date",
    'Time' : "kick_off",
    'HomeTeam' : "home_team",
    'AwayTeam' : "away_team",
    'FTHG' : "full_time_home_goals",
    'FTAG' : "full_time_away_goals",
    'FTR' : "full_time_match_result",
    'HTHG' : "half_time_home_goals",
    'HTAG' : "half_time_away_goals",
    'HTR' : "half_time_match_result",
    'HS' : "home_shots",
    'AS' : "away_shots",
    'HST' : "home_shots_on_target",
    'AST' : "away_shots_on_target",
    'HC' : "home_corners",
    'AC' : "away_corners",
    'HF' : "home_fouls",
    'AF' : "away_fouls",
    'HY' : "home_yellow_cards",
    'AY' : "away_yellow_cards",
    'HR' : "home_red_cards",
    'AR' : "away_red_cards",
}

In [156]:
bookies = {
    "Avg" : "market_average",
    "Max" : "market_maximum",
    "B365" : "bet365",
    "BW" : "bet_&_win",
    "PS" : "pinnacle"
}

bookies_cols = {}

for col in common_cols - set(non_bookies):

    for abbreviation, bookmaker in bookies.items():

        if not col.startswith(abbreviation):
            continue

        remainder = col[len(abbreviation):]

        name = bookmaker

        if remainder.startswith("C"):
            name += "_closing"
            remainder = remainder[1:]

        if remainder == "H":
            name += "_home_odds"
        elif remainder == "D":
            name += "_draw_odds"
        elif remainder == "A":
            name += "_away_odds"
        else:
            continue

        bookies_cols[col] = name

In [157]:
bookies_cols

{'MaxCD': 'market_maximum_closing_draw_odds',
 'AvgH': 'market_average_home_odds',
 'AvgCD': 'market_average_closing_draw_odds',
 'B365CD': 'bet365_closing_draw_odds',
 'AvgA': 'market_average_away_odds',
 'BWD': 'bet_&_win_draw_odds',
 'AvgCA': 'market_average_closing_away_odds',
 'PSCA': 'pinnacle_closing_away_odds',
 'BWCA': 'bet_&_win_closing_away_odds',
 'B365A': 'bet365_away_odds',
 'PSCH': 'pinnacle_closing_home_odds',
 'PSCD': 'pinnacle_closing_draw_odds',
 'BWCD': 'bet_&_win_closing_draw_odds',
 'AvgD': 'market_average_draw_odds',
 'MaxCA': 'market_maximum_closing_away_odds',
 'PSH': 'pinnacle_home_odds',
 'B365CA': 'bet365_closing_away_odds',
 'PSD': 'pinnacle_draw_odds',
 'MaxA': 'market_maximum_away_odds',
 'B365H': 'bet365_home_odds',
 'BWA': 'bet_&_win_away_odds',
 'B365D': 'bet365_draw_odds',
 'PSA': 'pinnacle_away_odds',
 'MaxD': 'market_maximum_draw_odds',
 'MaxCH': 'market_maximum_closing_home_odds',
 'BWH': 'bet_&_win_home_odds',
 'B365CH': 'bet365_closing_home_odds'